# POLITE — 2026-08-19 observation

Short, modular repaired-mount program: three low-polarization and three polarized V-band standards for an instrumental-polarization, response, and repeatability check. The repair is assumed successful: both mount axes enable and home, normal sidereal tracking is required, and the standard Mode 5 / gain 56 / offset 20 detector configuration is commanded and read back. The only acquisition path is `scripts/execute_night.py`; this notebook launches or inspects that runner and never opens the camera itself.

The before-dark cells come first, but are optional. Tonight's time-critical standard sequence should run first; take the calibration after science unless it has already completed. Do not run both full calibration cells.

## Session card

**ESTABLISHED:** target cooler setpoint −10 °C; detector defaults Mode 5 / gain 56 / offset 20; V-band HWP sequence 0°–157.5° in 22.5° steps; two frames at each angle. The six-star program is 96 light frames: HD 154445 uses the established 3 s exposure and the other five targets use 30 s, for 40.8 min of shutter time. Any clipping invalidates that target's complete cycle.

**PWI4 convention:** its displayed/API `altitude_degs` is zenith distance: 0° = zenith and 90° = horizon. The shed-safe PWI4 window is 3°–42°. Physical altitude for airmass/FITS is `90° − PWI4 value`.

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / 'pyproject.toml').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if not (ROOT / 'pyproject.toml').exists():
    raise RuntimeError('Could not locate the POLITE repository root.')
os.chdir(ROOT)

PYTHON = Path('/Users/blu3/miniforge3/envs/POLITE/bin/python')
if not PYTHON.exists():
    PYTHON = Path(sys.executable)

SESSION = '20260819'
PANEL_PROBE_PLAN = ROOT / 'night_plans/20260819_panel_probe.yaml'
CALIBRATION_PLAN = ROOT / 'night_plans/20260819_calibration.yaml'
SCIENCE_PLAN = ROOT / 'night_plans/20260819_standards.yaml'

def runner_command(plan: Path, *, subdir: str | None = None, run: bool = False) -> list[str]:
    command = [str(PYTHON), 'scripts/execute_night.py', str(plan)]
    if run:
        command.extend(['--run', '--setpoint', '-10', '--yes'])
    if subdir is not None:
        command.extend(['--subdir', subdir])
    return command

def preview_plan(plan: Path) -> None:
    subprocess.run(runner_command(plan), cwd=ROOT, check=True)

def run_plan(plan: Path, *, subdir: str) -> None:
    command = runner_command(plan, subdir=subdir, run=True)
    print('MOTION — repaired-night runner owns camera/EFW/HWP/mount:', ' '.join(command))
    subprocess.run(command, cwd=ROOT, check=True)

print('POLITE root:', ROOT)
print('Setpoint:', '-10 C')

## 1 · Optional panel calibration — defer until after standards

Use this section only if it is already complete before the time-critical standards. Otherwise proceed directly to §2 and take this calibration after science: it must not delay HD 154445 or HD 161056. The panel must be spatially uniform **and** effectively unpolarized; uniformity alone does not make a polarimetric flat valid.

These cells are independent of the science cell below. If the repair runs late, skip this section and return after the standard sequence; that is the preferred contingency, not a reason to delay science.

In [ ]:
# Read-only plan check; touches no hardware.
preview_plan(PANEL_PROBE_PLAN)

In [ ]:
# MOTION — four 1-s panel probes at HWP 0, 22.5, 45, and 67.5 deg.
# Run once, inspect for clipping and angle-dependent structure, then edit only
# panelFlatV.exp in 20260819_calibration.yaml before the full set.
# run_plan(PANEL_PROBE_PLAN, subdir='panel_probe')

In [ ]:
# Read-only plan check after choosing the panel exposure.
preview_plan(CALIBRATION_PLAN)

In [ ]:
# MOTION — run at most ONE full calibration block: this before-dark cell OR
# the after-science cell below. It captures V panel flats (10 per HWP angle),
# 20 biases, and 10 matching darks at both 3 s and 30 s through the opaque
# Dark slot.
# run_plan(CALIBRATION_PLAN, subdir='cal_before')

## 2 · Pointed standard-star ensemble

Run this six-star sequence immediately after the preamble: HD 154445 (polarized, setting), HD 161056 (polarized, setting), BD+32 3739 (low polarization, before zenith), HD 154892 (low polarization), HD 154345 (low polarization), then HD 183143 (polarized). Each target gets one complete eight-angle/two-frame V-band cycle. Do not add targets, repeat cycles, field rotations, dithers, or salvage overrides tonight.

The runner enables, homes, and checks both mount axes, then verifies the *post-slew* PWI4 zenith distance is inside 3°–42° before it permits the first frame. It is therefore safe to execute this section even if no calibration cell above was run.

In [ ]:
# Query only — no connect, home, slew, or capture. This is a sanity display;
# the runner repeats the same 3--42 deg check after its actual slew.
from obs_utils.config import default_sky_regions
from obs_utils.obs_math import zenith_distance_to_altitude
from obs_utils.pwi4_client import PWI4
from obs_utils.user_config import PWI4_CONFIG

pwi4 = PWI4(host=PWI4_CONFIG.host, port=PWI4_CONFIG.port)
status = pwi4.status()
z = float(status.mount.altitude_degs)
region, = default_sky_regions()
print(f'PWI4 zenith distance: {z:.2f} deg')
print(f'Conventional altitude: {zenith_distance_to_altitude(z):.2f} deg')
print(f'Allowed PWI4 range: {region.alt_min_deg:.0f}--{region.alt_max_deg:.0f} deg')
if region.alt_min_deg <= z <= region.alt_max_deg:
    print('Mount currently inside the observing window.')
else:
    print('Mount is currently outside the observing window (normal if parked); '
          'the runner verifies the target after its slew before capture.')

In [ ]:
# Read-only dry-run. Confirm the six targets, -10 C setpoint, HWP angles, and
# predicted mount actions before running the next cell.
preview_plan(SCIENCE_PLAN)

In [ ]:
# MOTION — abbreviated repaired-night standard program. This uses normal
# --mount auto behavior: both axes enable/home, every target slews, and PWI4
# must read back sidereal tracking before any frame. Do not add --mount off,
# --unpointed, or any --skip-*-check flag. The before-dark calibration cells
# need not have been run.
# run_plan(SCIENCE_PLAN, subdir='standards')

## 3 · Read-only science handoff

Run this after the science block to inspect FITS provenance and HWP coverage only. The instrumental q/u estimate, polarized response/position-angle comparison, and empirical source S/N require tracked paired-aperture photometry in the dated reduction notebook; do not infer them from whole-frame pixel statistics here. If any science frame clips, treat that target's complete HWP sequence as invalid and revise the plan before a replacement cycle.

In [ ]:
from obs_utils import live

SCIENCE_DIR = ROOT / 'FITSDATA' / SESSION / 'standards'
science_paths = sorted(SCIENCE_DIR.glob('*.fits'))
if not science_paths:
    print('No science files yet:', SCIENCE_DIR)
else:
    print('Science files:', len(science_paths))
    live.session_table(SCIENCE_DIR)
    live.hwp_coverage(SCIENCE_DIR)

## 4 · Optional after-science calibration

Run this only if the full calibration block was not taken before dark. It is intentionally the same plan and detector state, but writes to a different subdirectory. Panel flats, biases, and darks can therefore be deferred without changing or rerunning the science sequence.

In [ ]:
# MOTION — use only when cal_before was NOT run.
# run_plan(CALIBRATION_PLAN, subdir='cal_after')

## Closeout

The runner closes the camera/EFW/HWP itself after each block. Preserve the raw FITS, `block_manifest.jsonl`, and `pol_config.yaml` from each subdirectory before leaving. Record whether calibration was `cal_before` or `cal_after`, the final panel exposure, every target completed, any clipping, and the PWI4 zenith-distance read-back in the observing log.